# Chapter 6 — Neighborhoods and Manifolds

**Book alignment:** Embeddings From First Principles, Chapter 6

**Notebook role:** `ARTIFACT_REPLAY` — reads the frozen experiment artifact(s) this chapter cites and re-derives the numbers quoted in the prose (assertions fail if the artifact drifts).

**Question this notebook isolates:** Is local structure uniform? Specifically — does the
in-degree distribution (how often a point is *somebody's* nearest neighbour) become
right-skewed as dimension rises (a few **hubs**, many anti-hubs), and does that skew track
a model's anisotropy on RELATE while barely costing retrieval at this corpus size?

In [ ]:
from pathlib import Path
import json
import numpy as np

rng = np.random.default_rng(0)


def find_repo_root(start: Path) -> Path:
    for c in (start, *start.parents):
        if (c / "experiments" / "embeddings-from-first-principles" / "wave1").is_dir():
            return c
    raise RuntimeError("run from a checkout containing experiments/embeddings-from-first-principles")


ROOT = find_repo_root(Path.cwd().resolve())
EXP = ROOT / "experiments" / "embeddings-from-first-principles"


def art(wave: str, name: str) -> dict:
    return json.loads((EXP / wave / "artifacts" / name).read_text())

## 1. Hubness is a high-dimensional artifact — reproduce it

In [ ]:
def in_degree_skew(M, k=10):
    D = ((M[:, None] - M[None]) ** 2).sum(-1)
    nn = np.argsort(D, axis=1)[:, 1:k + 1]
    deg = np.bincount(nn.ravel(), minlength=len(M))
    m = deg.mean()
    return float(((deg - m) ** 3).mean() / deg.std() ** 3), int(deg.max())

for d in (2, 20, 200):
    pts = rng.standard_normal((600, d))
    skew, mx = in_degree_skew(pts)
    print(f"dim {d:3}:  in-degree skew {skew:+.2f}   max in-degree {mx}  (mean 10)")
low = in_degree_skew(rng.standard_normal((600, 2)))[0]
high = in_degree_skew(rng.standard_normal((600, 200)))[0]
assert high > low
print("\nlow-dim: every point ~= NN to k others. high-dim: a few points sink hundreds of lists")

## 2. On RELATE, the skew tracks anisotropy — and does not cost retrieval (Wave 1)

In [ ]:
hub = art("wave1", "hubness.json")["models"]
ani = art("wave1", "anisotropy.json")["models"]
print(f"{'model':14} {'in-deg skew':>11} {'origin':>7} {'R@10 all':>9} {'R@10 -hubs':>11}")
for m in hub:
    print(f"{m:14} {hub[m]['in_degree_skew']:>11.2f} {ani[m]['mean_random_cosine']:>7.2f}"
          f" {hub[m]['recall10_all']:>9.3f} {hub[m]['recall10_hubs_removed']:>11.3f}")

skews = [hub[m]["in_degree_skew"] for m in hub]
origins = [ani[m]["mean_random_cosine"] for m in hub]
r = np.corrcoef(skews, origins)[0, 1]
assert r > 0.8                                   # skew grows with the model's anisotropy
# removing the top-1% hubs barely moves Recall@10 on a 1,173-item index
deltas = [abs(hub[m]["recall10_all"] - hub[m]["recall10_hubs_removed"]) for m in hub]
assert max(deltas) < 0.01
print(f"\nskew vs anisotropy correlation r={r:.2f}; hub removal changes Recall@10 by <0.01")
print("hubness is real and structural; the 'hubs displace answers' cost needs a much larger index")

## What we earned

The unit of meaning is the *neighbourhood*, and neighbourhoods are lumpy: as dimension
rises, a handful of points become everyone's nearest neighbour. On RELATE the in-degree
skew rises with each model's anisotropy (r ≈ 0.9), and the hubs are the short, generic
sentences — but on a 1,173-item index, removing them does not change Recall@10. The
geometry is visible; the damage needs scale.

**Notebook 07 / Chapter 7** measures how many dimensions the representation actually uses —
and finds a 768-number vector living in far fewer.